# 01 — Raw Data Audit and Player-Season Panel

**Project:** CSMPR MSc dissertation — next-season football player performance prediction  
**Source dataset:** *Top 5 League Football Player Stats (2017–2025)*, Kaggle  
**Source URL:** https://www.kaggle.com/datasets/emrey3lmaz/top-5-league-football-player-stats-2017-2025  
**Input:** `data/raw/Dataset_Original.csv`  
**Output:** `data/processed/Dataset_PlayerSeason_Modelling.csv`

This notebook is the first executable stage of the analysis. It imports the original semicolon-delimited dataset, documents and corrects the confirmed scraper fan-out duplication, constructs a stable player identifier, audits coverage and missingness, and aggregates multi-club records to a single player-season row. The resulting player-season panel is the sole input to target construction in Notebook 02.

## 0. Setup

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks": ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.analysis_utils import paths
P = paths(ROOT)

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

RAW_PATH = P["raw"] / "Dataset_Original.csv"
SEASON_ORDER = ['1718','1819','1920','2021','2122','2223','2324','2425']
ID_STR_COLS = ['league','team','player','nation_','pos_']


## 1. Load and inspect the raw file

The source file is semicolon-delimited and uses **comma decimals** (European format, e.g. `16,6` for 16.6), so it must be loaded as strings first and converted column-by-column — a naive `pd.read_csv` would silently mis-type most numeric columns.

In [2]:
raw = pd.read_csv(RAW_PATH, sep=';', low_memory=False, dtype=str)
print("Raw shape:", raw.shape)
print("\nLeague counts:\n", raw['league'].value_counts())
print("\nSeason counts:\n", raw['season'].value_counts().sort_index())
raw.head(3)


Raw shape: (22929, 178)

League counts:
 league
ITA-Serie A           5336
ESP-La Liga           4692
FRA-Ligue 1           4543
ENG-Premier League    4360
GER-Bundesliga        3998
Name: count, dtype: int64

Season counts:
 season
1718    3202
1819    2656
1920    2732
2021    2822
2122    2922
2223    2889
2324    2852
2425    2854
Name: count, dtype: int64


,league,season,team,player,nation_,pos_,age_,born_,Playing Time_MP,Playing Time_Starts,...,Performance_Crs,Performance_Fld,Performance_Fls,Performance_Int,Performance_OG,Performance_Off,Performance_PKcon,Performance_PKwon,Performance_Recov,Performance_TklW
0,ENG-Premier League,1718,Arsenal,Aaron Ramsey,WAL,MF,26,1990,24,21,...,17,33,28,24,0,0,0,2,128,27
1,ENG-Premier League,1718,Arsenal,Ainsley Maitland-Niles,ENG,"DF,MF",19,1997,15,8,...,20,5,10,12,0,0,1,0,73,17
2,ENG-Premier League,1718,Arsenal,Alex Iwobi,NGA,"MF,FW",21,1996,26,22,...,20,12,11,7,0,1,0,0,110,15


## 2. Fix the scraper duplication bug

An early check on duplicate `(player, season, team)` keys turned up exactly one anomalous case: **Emanuele Torrasi, AC Milan, 2017/18 — 512 rows** instead of 1. Inspecting those 512 rows shows every column has **at most one distinct non-null value**; the rest are `NaN`. This is a classic many-to-many join fan-out (FBref's ~15 separate stat tables joined by player name, where the join key produced a Cartesian product for this one player), not genuine data.

The fix is a generic coalesce — `groupby(...).first()` takes the first *non-null* value per column within each group — applied to every `(player, season, team, league)` group, which only affects this one case but is written generically in case the same bug recurs elsewhere in future scrapes.

In [3]:
dupe_check = raw.groupby(['player','season','team']).size()
print("player+season+team keys with >1 row:", (dupe_check>1).sum())
print(dupe_check[dupe_check>1])


player+season+team keys with >1 row: 1
player            season  team 
Emanuele Torrasi  1718    Milan    512
dtype: int64


In [4]:
torrasi = raw[raw['player']=='Emanuele Torrasi'].reset_index(drop=True)
nun = torrasi.nunique(dropna=True)
print("Columns with >1 distinct NON-NULL value across the 512 rows:", (nun>1).sum(), "of", len(nun))
print(nun[nun>1])  # only Subs_unSub varies -> confirms fan-out, not genuine duplicate seasons


Columns with >1 distinct NON-NULL value across the 512 rows: 1 of 178
Subs_unSub    2
dtype: int64


## 3. Type conversion

Convert every numeric column (comma-decimal strings) to proper floats, leaving the five identifier columns as strings.

In [5]:
df = raw.copy()

def to_float(s):
    return pd.to_numeric(s.str.replace(',', '.', regex=False), errors='coerce')

numeric_cols = [c for c in df.columns if c not in ID_STR_COLS + ['season']]
conv = {}
for c in numeric_cols:
    conv[c] = to_float(df[c]) if df[c].dtype == object else pd.to_numeric(df[c], errors='coerce')
df = pd.concat([df[ID_STR_COLS + ['season']], pd.DataFrame(conv, index=df.index)], axis=1)
df = df[raw.columns]  # restore original column order

df['season'] = df['season'].astype(str)
for c in ID_STR_COLS:
    df[c] = df[c].astype(str).str.strip()

print(df.dtypes.value_counts())


float64    151
int64       21
object       6
Name: count, dtype: int64


## 4. Player identifier

FBref exposes no numeric player ID in this scrape. Using `player` name alone collides for **namesakes** — genuinely different real people who share a name (verified below by checking for players with more than one distinct birth year under the same name). `player_id = player name + birth year` resolves nearly all of these.

In [6]:
df['player_id'] = df['player'] + "||" + df['born_'].fillna(-1).astype(int).astype(str).replace('-1', 'NA')

name_born = df.groupby('player')['born_'].nunique()
namesakes = name_born[name_born > 1]
print("Names with >1 distinct birth year (genuine namesake collisions):", len(namesakes))
print("Unique player_id (name+birth year):", df['player_id'].nunique(), " vs unique player name alone:", df['player'].nunique())

pid_nation = df.groupby('player_id')['nation_'].nunique(dropna=True)
print("player_id groups with inconsistent nation_ (residual identity noise):", (pid_nation>1).sum())


Names with >1 distinct birth year (genuine namesake collisions): 48
Unique player_id (name+birth year): 7185  vs unique player name alone: 7136
player_id groups with inconsistent nation_ (residual identity noise): 1


## 5. Coalesce the fan-out bug generically

Group by the full identity key and take the first non-null value per column — this collapses the 512 Torrasi rows to 1, and would do the same for any future occurrence of the same bug.

In [7]:
key = ['player_id', 'season', 'team', 'league']
grp_sizes_before = df.groupby(key).size()
n_bugged_groups = (grp_sizes_before > 1).sum()
n_bugged_rows = grp_sizes_before[grp_sizes_before > 1].sum()

df_pst = df.groupby(key, as_index=False).first()

print("Raw rows:", len(raw))
print("Fan-out/duplicate-key groups found:", n_bugged_groups, "covering", n_bugged_rows, "raw rows")
print("Rows after coalescing to one row per player-season-team-league:", len(df_pst))


Raw rows: 22929
Fan-out/duplicate-key groups found: 1 covering 512 raw rows
Rows after coalescing to one row per player-season-team-league: 22418


## 6. Audit 1 — Seasons and leagues covered

In [8]:
df_pst['season'] = pd.Categorical(df_pst['season'], categories=SEASON_ORDER, ordered=True)

rows_by_season_league = df_pst.groupby(['season','league'], observed=True).size().unstack(fill_value=0).reindex(SEASON_ORDER)
rows_by_season_league['Total_rows'] = rows_by_season_league.sum(axis=1)

players_by_season_league = df_pst.groupby(['season','league'], observed=True)['player_id'].nunique().unstack(fill_value=0).reindex(SEASON_ORDER)
players_by_season_league['Total_players'] = df_pst.groupby('season', observed=True)['player_id'].nunique().reindex(SEASON_ORDER)

print("Player-season-team rows by season x league:")
display(rows_by_season_league)
print("\nUnique players by season x league:")
display(players_by_season_league)


Player-season-team rows by season x league:


league,ENG-Premier League,ESP-La Liga,FRA-Ligue 1,GER-Bundesliga,ITA-Serie A,Total_rows
season,,,,,,
1718,529,573,553,484,552,2691
1819,508,544,561,474,569,2656
1920,522,570,542,498,600,2732
2021,532,582,584,505,619,2822
2122,546,617,604,523,632,2922
2223,569,596,606,515,603,2889
2324,580,609,540,507,616,2852
2425,574,601,553,492,634,2854



Unique players by season x league:


league,ENG-Premier League,ESP-La Liga,FRA-Ligue 1,GER-Bundesliga,ITA-Serie A,Total_players
season,,,,,,
1718,515,557,540,472,533,2566
1819,505,531,548,468,543,2556
1920,515,556,531,487,572,2613
2021,524,570,574,496,594,2704
2122,537,603,590,511,608,2791
2223,554,583,584,506,577,2724
2324,570,598,524,494,590,2710
2425,562,589,542,481,599,2707


## 7. Audit 2 — Player and player-season counts

In [9]:
n_player_season_team_rows = len(df_pst)
n_player_seasons = df_pst.groupby(['player_id','season'], observed=True).ngroups
n_unique_players = df_pst['player_id'].nunique()
sizes = df_pst.groupby(['player_id','season'], observed=True).size()
n_multi_team_seasons = int((sizes > 1).sum())

cross_league = 0
for (pid, season), g in df_pst.groupby(['player_id','season'], observed=True):
    if len(g) > 1 and g['league'].nunique() > 1:
        cross_league += 1

summary = pd.DataFrame({
    'metric': ['player-season-team rows (post-cleaning)',
               'unique player-seasons -> final modelling dataset row count',
               'unique players (player_id) across all seasons',
               'player-seasons split across >1 club (in-season transfer)',
               'of which split across >1 league'],
    'value': [n_player_season_team_rows, n_player_seasons, n_unique_players, n_multi_team_seasons, cross_league]
})
summary


,metric,value
0,player-season-team rows (post-cleaning),22418
1,unique player-seasons -> final modelling datas...,21371
2,unique players (player_id) across all seasons,7185
3,player-seasons split across >1 club (in-season...,1044
4,of which split across >1 league,461


## 8. Audit 3 — Missingness by variable and season

Missingness is checked both overall and by season, to distinguish **structural** missingness (e.g. a shot-based ratio is undefined when a player took zero shots) from **era-based** gaps (e.g. a stat category FBref didn't track in early seasons) or random data loss.

In [10]:
stat_cols = [c for c in df_pst.columns if c not in
             ['league','team','player','nation_','pos_','season','player_id','age_','born_']]

miss_overall = df_pst[stat_cols].isna().mean().sort_values(ascending=False) * 100
miss_by_season = df_pst.groupby('season', observed=True)[stat_cols].apply(lambda g: g.isna().mean()*100).reindex(SEASON_ORDER)

print("Columns with 0% missing overall:", (miss_overall==0).sum(), "/", len(miss_overall))
print("\nTop 15 most-missing columns overall:")
display(miss_overall.head(15))


Columns with 0% missing overall: 30 / 170

Top 15 most-missing columns overall:


Standard_G/SoT         30.747614
Subs_Mn/Sub            22.147382
Starts_Mn/Start        18.248729
Expected_npxG/Sh       18.239807
Standard_Dist          18.239807
Standard_G/Sh          18.230886
Standard_SoT%          18.230886
Take-Ons_Succ%         15.411723
Take-Ons_Tkld%         15.411723
Challenges_Tkl%        11.526452
Aerial Duels_Won%       7.302168
Long_Cmp%               6.021947
Medium_Cmp%             2.332947
Short_Cmp%              1.570167
Team Success_On-Off     0.838612
dtype: float64

In [11]:
# Is missingness era-based (early seasons only) or structural (zero-denominator ratios)?
structural = [c for c in stat_cols if miss_by_season[c].iloc[0] > 90 and miss_by_season[c].iloc[-1] < 10]
print("Columns with an early-season-only ('era') missingness pattern:", len(structural))

# Test the structural-zero-denominator hypothesis directly:
print("\n% missing Standard_G/SoT, goalkeepers vs outfield:")
print(df_pst.groupby(df_pst['pos_'].str.contains('GK', na=False))['Standard_G/SoT'].apply(lambda s: s.isna().mean()*100))
print("\n% missing Starts_Mn/Start, non-starters vs starters:")
print(df_pst.groupby(df_pst['Playing Time_Starts']==0)['Starts_Mn/Start'].apply(lambda s: s.isna().mean()*100))


Columns with an early-season-only ('era') missingness pattern: 0

% missing Standard_G/SoT, goalkeepers vs outfield:
pos_
False    25.298651
True     98.974668
Name: Standard_G/SoT, dtype: float64

% missing Starts_Mn/Start, non-starters vs starters:
Playing Time_Starts
False      7.237941
True     100.000000
Name: Starts_Mn/Start, dtype: float64


**Finding:** no column shows the tested early-versus-late-season coverage pattern. The largest concentrations of missing values are consistent with structural undefined ratios, including shot-based measures for goalkeepers and minutes-per-start for non-starters. This supports treating missingness as predominantly structural at the audit stage while retaining model-stage missing values for training-only handling rather than assuming that every missing cell has the same cause.

## 9. Audit 4 — Minutes distribution

In [12]:
ps_min = df_pst.groupby(['player_id','season'], observed=True)['Playing Time_Min'].sum().reset_index()

print(ps_min['Playing Time_Min'].describe(percentiles=[.05,.1,.25,.5,.75,.9,.95]))

bins = [-1,0,90,450,900,1800,2700,3420]
labels = ['0','1-90','91-450','451-900','901-1800','1801-2700','2701+']
ps_min['bucket'] = pd.cut(ps_min['Playing Time_Min'], bins=bins, labels=labels)
bucket_tab = ps_min['bucket'].value_counts().reindex(labels)
print("\nMinutes bucket distribution:")
display(pd.DataFrame({'n_player_seasons': bucket_tab, 'pct': (bucket_tab/len(ps_min)*100).round(2)}))


count    21371.000000
mean      1327.080202
std        979.073592
min          1.000000
5%          18.000000
10%         72.000000
25%        408.000000
50%       1248.000000
75%       2125.000000
90%       2735.000000
95%       2988.000000
max       3583.000000
Name: Playing Time_Min, dtype: float64

Minutes bucket distribution:


,n_player_seasons,pct
bucket,,
0,0,0.00
1-90,2624,12.28
91-450,2993,14.00
451-900,2869,13.42
901-1800,5630,26.34
1801-2700,4966,23.24
2701+,2288,10.71


## 10. Audit 5 — Players appearing in consecutive seasons (retention)

In [13]:
idx = df_pst.groupby(['player_id','season'], observed=True)['Playing Time_Min'].idxmax()
ps_primary = df_pst.loc[idx, ['player_id','season','team','league','Playing Time_Min']].reset_index(drop=True)
ps_primary['season'] = pd.Categorical(ps_primary['season'], categories=SEASON_ORDER, ordered=True)

season_sets = {s: set(ps_primary.loc[ps_primary['season']==s,'player_id']) for s in SEASON_ORDER}

rows = []
for i in range(len(SEASON_ORDER)-1):
    s0, s1 = SEASON_ORDER[i], SEASON_ORDER[i+1]
    a, b = season_sets[s0], season_sets[s1]
    retained, left, entered = len(a & b), len(a - b), len(b - a)
    rows.append({'season_t':s0, 'season_t+1':s1, 'n_in_t':len(a), 'n_in_t+1':len(b),
                 'retained_both':retained, 'left_after_t':left, 'new_entrants_in_t+1':entered,
                 'retention_rate_pct': round(retained/len(a)*100,2)})
retention = pd.DataFrame(rows)
retention


,season_t,season_t+1,n_in_t,n_in_t+1,retained_both,left_after_t,new_entrants_in_t+1,retention_rate_pct
0,1718,1819,2566,2556,1799,767,757,70.11
1,1819,1920,2556,2613,1795,761,818,70.23
2,1920,2021,2613,2704,1888,725,816,72.25
3,2021,2122,2704,2791,1905,799,886,70.45
4,2122,2223,2791,2724,1862,929,862,66.71
5,2223,2324,2724,2710,1830,894,880,67.18
6,2324,2425,2710,2707,1829,881,878,67.49


This table also answers **Audit 8 (players entering/leaving the sample)**: `new_entrants_in_t+1` and `left_after_t` are exactly that.

## 11. Audit 6 — Positional distribution

In [14]:
raw_pos = df_pst['pos_'].value_counts()
print("Raw pos_ code distribution (top 10):")
display(raw_pos.head(10))

df_pst['pos_primary'] = df_pst['pos_'].str.split(',').str[0]
prim = df_pst['pos_primary'].value_counts()
print("\nPrimary position (first-listed code):")
display(prim)
print("\nShare with a multi-position code:", round(df_pst['pos_'].str.contains(',', na=False).mean()*100, 2), "%")


Raw pos_ code distribution (top 10):


pos_
DF       6736
MF       4746
FW       3024
FW,MF    2371
MF,FW    1986
GK       1658
DF,MF     795
MF,DF     546
DF,FW     374
FW,DF     181
Name: count, dtype: int64


Primary position (first-listed code):


pos_primary
DF     7905
MF     7278
FW     5576
GK     1658
nan       1
Name: count, dtype: int64


Share with a multi-position code: 27.89 %


## 12. Audit 7 — Transfers between seasons

Club/league changes among players retained across consecutive seasons (using each player's primary/highest-minutes club per season).

In [15]:
transfer_rows = []
for i in range(len(SEASON_ORDER)-1):
    s0, s1 = SEASON_ORDER[i], SEASON_ORDER[i+1]
    p0 = ps_primary[ps_primary['season']==s0][['player_id','team','league']].rename(columns={'team':'team_t','league':'league_t'})
    p1 = ps_primary[ps_primary['season']==s1][['player_id','team','league']].rename(columns={'team':'team_t1','league':'league_t1'})
    m = p0.merge(p1, on='player_id', how='inner')
    m['club_changed'] = m['team_t'] != m['team_t1']
    m['league_changed'] = m['league_t'] != m['league_t1']
    transfer_rows.append({
        'season_t':s0,'season_t+1':s1,'n_retained':len(m),
        'club_changed_pct':round(m['club_changed'].mean()*100,2),
        'league_changed_pct':round(m['league_changed'].mean()*100,2),
    })
pd.DataFrame(transfer_rows)


,season_t,season_t+1,n_retained,club_changed_pct,league_changed_pct
0,1718,1819,1799,27.68,10.17
1,1819,1920,1795,27.13,10.03
2,1920,2021,1888,22.40,8.69
3,2021,2122,1905,22.10,8.71
4,2122,2223,1862,28.14,11.49
5,2223,2324,1830,28.14,12.62
6,2324,2425,1829,27.99,11.97


## 13. Audit 9 — Duplicate or inconsistent player identifiers

In [16]:
issues = {
    'scraper_fanout_duplicate_rows_fixed': int(n_bugged_rows),
    'distinct_names_with_multiple_birth_years (namesakes)': int(len(namesakes)),
    'player_ids_with_inconsistent_nation': int((pid_nation > 1).sum()),
    'rows_missing_born_': int(df_pst['born_'].isna().sum()),
    'rows_missing_nation_': int(df_pst['nation_'].isna().sum()),
    'rows_missing_pos_': int(df_pst['pos_'].isna().sum()),
    'residual_duplicate_keys_after_cleaning': int(df_pst.duplicated(subset=['player_id','season','team','league']).sum()),
}
pd.Series(issues)


scraper_fanout_duplicate_rows_fixed                     512
distinct_names_with_multiple_birth_years (namesakes)     48
player_ids_with_inconsistent_nation                       1
rows_missing_born_                                        3
rows_missing_nation_                                      0
rows_missing_pos_                                         0
residual_duplicate_keys_after_cleaning                    0
dtype: int64

## 14. Build the modelling dataset — one row per player-season

1,044 player-seasons involve more than one club (in-season transfer). To produce **one row per player-season** as required for modelling, these are aggregated:

- **Counting stats** (goals, minutes, passes attempted, etc.) are **summed** across clubs.
- **Ratio/percentage columns** (e.g. pass completion %, shot conversion %) are **recomputed from the summed numerator/denominator** — never averaged directly, which would bias toward the club with fewer attempts.
- **Per-90 columns** are recomputed as summed raw stat ÷ summed 90s.
- A handful of columns with no clean recompute formula (e.g. `Team Success_On-Off`, `Aerial Duels_Won%`) fall back to a **minutes-weighted mean** of the original per-row values.
- `player`, `nation_`, `pos_`, `age_`, `born_` are taken from the player's highest-minutes club that season.
- `team_primary`/`league_primary` = highest-minutes club; `teams_all`/`leagues_all` list every club played for; `transferred_clubs`/`transferred_leagues` flag multi-club/cross-league seasons.

In [17]:
ID_COLS = ['league','team','player','nation_','pos_','player_id']
non_stat = ID_COLS + ['season','age_','born_']
stat_cols2 = [c for c in df_pst.columns if c not in non_stat + ['pos_primary']]

ratio_formula = {
    'Standard_SoT%':      ('Standard_SoT','Standard_Sh', 100),
    'Standard_G/Sh':       ('Performance_Gls','Standard_Sh', 1),
    'Standard_G/SoT':      ('Performance_Gls','Standard_SoT', 1),
    'Expected_npxG/Sh':    ('Expected_npxG','Standard_Sh', 1),
    'Long_Cmp%':           ('Long_Cmp','Long_Att', 100),
    'Medium_Cmp%':         ('Medium_Cmp','Medium_Att', 100),
    'Short_Cmp%':          ('Short_Cmp','Short_Att', 100),
    'Total_Cmp%':          ('Total_Cmp','Total_Att', 100),
    'Challenges_Tkl%':     ('Challenges_Tkl','Challenges_Att', 100),
    'Take-Ons_Succ%':      ('Take-Ons_Succ','Take-Ons_Att', 100),
    'Take-Ons_Tkld%':      ('Take-Ons_Tkld','Take-Ons_Att', 100),
    'Playing Time_Mn/MP':  ('Playing Time_Min','Playing Time_MP', 1),
}
diff_formula = {
    'Team Success_+/-':          ('Team Success_onG','Team Success_onGA'),
    'Team Success (xG)_xG+/-':   ('Team Success (xG)_onxG','Team Success (xG)_onxGA'),
}
per90_formula = {
    'GCA_GCA90': ('GCA_GCA','90s_'), 'SCA_SCA90': ('SCA_SCA','90s_'),
    'Standard_Sh/90': ('Standard_Sh','90s_'), 'Standard_SoT/90': ('Standard_SoT','90s_'),
    'Per 90 Minutes_Gls': ('Performance_Gls','90s_'), 'Per 90 Minutes_Ast': ('Performance_Ast','90s_'),
    'Per 90 Minutes_G+A': ('Performance_G+A','90s_'), 'Per 90 Minutes_G-PK': ('Performance_G-PK','90s_'),
    'Per 90 Minutes_xG': ('Expected_xG','90s_'), 'Per 90 Minutes_xAG': ('Expected_xAG','90s_'),
    'Per 90 Minutes_npxG': ('Expected_npxG','90s_'),
}
composite_per90 = {
    'Per 90 Minutes_G+A-PK': (['Performance_G-PK','Performance_Ast'], '90s_'),
    'Per 90 Minutes_xG+xAG': (['Expected_xG','Expected_xAG'], '90s_'),
    'Per 90 Minutes_npxG+xAG': (['Expected_npxG','Expected_xAG'], '90s_'),
}
special_weighted_mean = {
    'Aerial Duels_Won%', 'Playing Time_Min%', 'Starts_Mn/Start', 'Subs_Mn/Sub',
    'Team Success_On-Off', 'Team Success (xG)_On-Off', 'Team Success_PPM',
    'Team Success_+/-90', 'Team Success (xG)_xG+/-90',
}
special_cols = set(ratio_formula) | set(diff_formula) | set(per90_formula) | set(composite_per90) | special_weighted_mean
sum_cols = [c for c in stat_cols2 if c not in special_cols]
assert set(sum_cols) | special_cols == set(stat_cols2)
print(len(sum_cols), "summed columns,", len(special_cols), "recomputed/derived columns")


133 summed columns, 37 recomputed/derived columns


In [18]:
g = df_pst.groupby(['player_id','season'], observed=True)
agg_sum = g[sum_cols].sum(min_count=1)

idx_max = df_pst.groupby(['player_id','season'], observed=True)['Playing Time_Min'].idxmax()
context = df_pst.loc[idx_max, ['player_id','season','player','nation_','pos_','age_','born_','team','league','Playing Time_Min']].set_index(['player_id','season'])
context = context.rename(columns={'team':'team_primary','league':'league_primary','Playing Time_Min':'min_in_primary_team'})

lists = g.agg(teams_all=('team', lambda s: '; '.join(sorted(set(s)))),
              leagues_all=('league', lambda s: '; '.join(sorted(set(s)))),
              n_team_rows=('team','size'))
lists['transferred_clubs'] = lists['n_team_rows'] > 1
lists['transferred_leagues'] = lists['leagues_all'].str.contains(';')

out = pd.concat([context, lists, agg_sum], axis=1)

for col,(num,den,mult) in ratio_formula.items():
    out[col] = np.where(out[den]>0, out[num]/out[den]*mult, np.nan)
for col,(a,b) in diff_formula.items():
    out[col] = out[a]-out[b]
for col,(num,den) in per90_formula.items():
    out[col] = np.where(out[den]>0, out[num]/out[den], np.nan)
for col,(nums,den) in composite_per90.items():
    numsum = sum(out[n] for n in nums)
    out[col] = np.where(out[den]>0, numsum/out[den], np.nan)

# vectorised minutes-weighted mean fallback for the remaining special columns
w = df_pst['Playing Time_Min'].clip(lower=0)
wv = pd.DataFrame({'player_id': df_pst['player_id'], 'season': df_pst['season']})
for col in special_weighted_mean:
    v = df_pst[col]; valid = v.notna()
    wv[f'__num_{col}'] = np.where(valid, v.fillna(0)*w, 0.0)
    wv[f'__den_{col}'] = np.where(valid, w, 0.0)
sums = wv.groupby(['player_id','season'], observed=True).sum(numeric_only=True)
for col in special_weighted_mean:
    out[col] = np.where(sums[f'__den_{col}']>0, sums[f'__num_{col}']/sums[f'__den_{col}'], np.nan)

out = out.reset_index()
season_map = {'1718':'2017/18','1819':'2018/19','1920':'2019/20','2021':'2020/21','2122':'2021/22','2223':'2022/23','2324':'2023/24','2425':'2024/25'}
out['season_label'] = out['season'].map(season_map)

front = ['player_id','player','season','season_label','born_','nation_','age_','pos_',
         'team_primary','league_primary','teams_all','leagues_all','transferred_clubs','transferred_leagues',
         'n_team_rows','min_in_primary_team']
out = out[front + [c for c in out.columns if c not in front]]
print("Final modelling dataset:", out.shape)
out.head(3)


Final modelling dataset: (21371, 186)


,player_id,player,season,season_label,born_,nation_,age_,pos_,team_primary,league_primary,...,Per 90 Minutes_npxG+xAG,Team Success (xG)_xG+/-90,Team Success_+/-90,Team Success_PPM,Team Success_On-Off,Subs_Mn/Sub,Playing Time_Min%,Team Success (xG)_On-Off,Aerial Duels_Won%,Starts_Mn/Start
0,Aaron Ciammaglichella||2005,Aaron Ciammaglichella,2425,2024/25,2005.0,ITA,19.0,MF,Torino,ITA-Serie A,...,NaN,9.23,0.00,3.00,0.16,1.0,0.0,9.65,NaN,NaN
1,Aaron Connolly||2000,Aaron Connolly,1920,2019/20,2000.0,IRL,19.0,FW,Brighton,ENG-Premier League,...,0.321429,-0.11,-0.36,1.13,0.06,26.0,36.8,0.23,31.0,72.0
2,Aaron Connolly||2000,Aaron Connolly,2021,2020/21,2000.0,IRL,20.0,FW,Brighton,ENG-Premier League,...,0.477273,0.99,-0.68,0.88,-0.68,23.0,23.1,0.75,18.4,68.0


## 15. Verification

Cross-check totals between the raw file and the final aggregated dataset — they must reconcile exactly once the known 512-row scraper artefact is accounted for.

In [19]:
print("Modelling dataset unique player_id+season:", out.groupby(['player_id','season']).ngroups, "== rows:", len(out))
print("Any duplicate keys:", out.duplicated(subset=['player_id','season']).sum())

raw_min_all = pd.to_numeric(raw['Playing Time_Min'].str.replace(',','.'), errors='coerce').sum()
print("\nRaw total minutes (incl. 511 extra fan-out rows @ 6 min each):", raw_min_all)
print("Modelling dataset total minutes:                                ", out['Playing Time_Min'].sum())
print("Expected difference (511 x 6 min):", raw_min_all - out['Playing Time_Min'].sum(), "(should be 3066)")

raw_gls_all = pd.to_numeric(raw['Performance_Gls'].str.replace(',','.'), errors='coerce').sum()
print("\nRaw total goals:", raw_gls_all, " Modelling dataset total goals:", out['Performance_Gls'].sum())


Modelling dataset unique player_id+season: 21371 == rows: 21371
Any duplicate keys: 0

Raw total minutes (incl. 511 extra fan-out rows @ 6 min each): 28364097
Modelling dataset total minutes:                                 28361031
Expected difference (511 x 6 min): 3066 (should be 3066)

Raw total goals: 38883  Modelling dataset total goals: 38883


/tmp/ipykernel_4043/503272057.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print("Modelling dataset unique player_id+season:", out.groupby(['player_id','season']).ngroups, "== rows:", len(out))


## 16. Save outputs

In [20]:
out.to_csv(P["processed"] / str(P["processed"] / "Dataset_PlayerSeason_Modelling.csv"), index=False)
print("Saved Dataset_PlayerSeason_Modelling.csv:", out.shape)


Saved Dataset_PlayerSeason_Modelling.csv: (21371, 186)


## 17. Report-ready audit exports

The audit quantities used later in the dissertation are saved explicitly rather than being reconstructed from notebook display output. These tables provide traceable counts for source rows, aggregation, identity checks, season retention and transfer patterns.

In [21]:
# Export report-ready audit summaries used by the methodology/results chapters.
audit_summary = pd.DataFrame({
    'metric': [
        'raw_rows','raw_variables','fanout_rows_identified','player_season_team_rows_after_coalescing',
        'final_unique_player_seasons','unique_players','multi_club_player_seasons',
        'cross_league_multi_club_player_seasons','residual_duplicate_player_season_team_keys'
    ],
    'value': [
        len(raw), raw.shape[1], n_bugged_rows, len(df_pst), len(out), n_unique_players,
        n_multi_team_seasons, cross_league,
        int(df_pst.duplicated(subset=['player_id','season','team','league']).sum())
    ]
})
identity_summary = pd.DataFrame({'metric': list(issues.keys()), 'value': list(issues.values())})
transfer_summary = pd.DataFrame(transfer_rows)

audit_summary.to_csv(P['tables']/'01_data_audit_summary.csv', index=False)
identity_summary.to_csv(P['tables']/'01_identity_audit_summary.csv', index=False)
retention.to_csv(P['tables']/'01_season_retention_summary.csv', index=False)
transfer_summary.to_csv(P['tables']/'01_season_to_season_transfer_summary.csv', index=False)
rows_by_season_league.to_csv(P['tables']/'01_rows_by_season_league.csv')
players_by_season_league.to_csv(P['tables']/'01_players_by_season_league.csv')
print('Saved report-ready data-audit tables to', P['tables'])

Saved report-ready data-audit tables to /mnt/data/CSMPR_Frozen_Analysis_Final/results/tables


## Summary of findings

- **Coverage:** five leagues across eight seasons (2017/18–2024/25).
- **Confirmed scraper defect:** one player-season-team key expanded to 512 complementary rows; coalescing reduced the source to 22,418 player-season-team rows with no residual duplicate key.
- **Identity handling:** player name plus birth year resolves the observed same-name collisions while leaving the small residual identity boundary documented.
- **Multi-club seasons:** aggregation uses statistic-appropriate rules rather than simple averaging.
- **Missingness:** no tested early-versus-late coverage gap was found; prominent patterns are consistent with structural undefined ratios and are left for training-only handling downstream.
- **Final panel:** 21,371 unique player-season observations representing 7,185 player identities.

## Stage conclusion

This stage produces the one-row-per-player-season panel used by every subsequent analysis. Duplicate-key, identity, aggregation-reconciliation and row-count assertions are executed before the dataset is written. Later notebooks consume this output directly and do not repeat the cleaning or aggregation decisions.